# TestFeedback Excel 정제 및 반별 저장

정기평가 원본 Excel인 **파일A**를 정제하고, 필요한 차수에는 **파일B**의 이전점수를 학번으로 결합한 뒤 반별 `.xlsx` 파일을 생성합니다.

이 노트북의 범위는 다음까지입니다.

- 파일A 자동 검색 및 A1 시험 차수 파싱
- 제1·2·4·5차 일반평가와 제3·6차 누적평가 정제
- 제2·4·5차 이전점수 결합
- 학번 문자열 및 앞자리 0 보존
- 학급명별 분리, 학생명 오름차순 정렬
- `DataClean` 폴더에 반별 Excel 저장

피드백 TXT 생성은 포함하지 않습니다.


## 실행 전 폴더 구조

```text
C:\Users\9191h\Desktop\TestFeedback
├─ Source
│  ├─ 현재 처리할 파일A.xlsx       # Source 바로 아래에는 .xlsx 1개만 둡니다.
│  └─ Legacy
│     └─ 이전 시험 파일B.xlsx
└─ DataClean                      # 반별 결과 저장 위치
```

필요한 패키지는 `pandas`, `openpyxl`입니다. 설치되지 않았다면 새 셀에서 `%pip install pandas openpyxl`을 한 번 실행하세요.

`DataClean`에 같은 이름의 파일이 있으면 기본 설정상 덮어씁니다. 현재 시험에 없는 반의 예전 파일은 자동 삭제하지 않습니다.


## 기본 설정과 공통 유틸리티

필요한 패키지, 고정 열 순서, 값 변환 및 검증 도구입니다.


In [1]:
# %pip install pandas openpyxl

In [2]:
from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass
from numbers import Integral, Real
from pathlib import Path
from typing import Any

import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


AREA_COLUMNS = [
    "어휘 파악",
    "핵심 요지 파악",
    "내용 파악",
    "문맥 파악",
    "구조 파악",
    "추론",
]
NORMAL_BASE_COLUMNS = ["학급명", "학생명", "학번", "점수", *AREA_COLUMNS]
TEACHER_SUFFIX = "-임서영T"
TEST_NUMBER_PATTERN = re.compile(r"제\s*(\d+)\s*차")
YEAR_PATTERN = re.compile(r"(\d{4})\s*년")
INVALID_FILENAME_CHARS = re.compile(r'[<>:"/\\|?*]')


class DataValidationError(ValueError):
    """입력 Excel의 구조나 값이 합의된 규칙과 다를 때 발생합니다."""


@dataclass(frozen=True)
class SourceMetadata:
    file_path: Path
    a1_text: str
    a2_text: str
    sheet_title: str
    test_year: int
    test_number: int


@dataclass
class CleaningResult:
    metadata: SourceMetadata
    cleaned_data: pd.DataFrame
    output_files: list[Path]


def _is_blank(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def _normalize_text(value: Any) -> str | None:
    if _is_blank(value):
        return None
    return re.sub(r"\s+", " ", str(value).strip())


def _normalize_student_id(value: Any) -> str | None:
    """학번은 언제나 문자열로 유지하며 앞자리 0을 보존합니다."""
    if _is_blank(value):
        return None
    if isinstance(value, bool):
        raise DataValidationError("학번에 논리값이 들어 있습니다.")
    if isinstance(value, Integral):
        return str(int(value))
    if isinstance(value, Real):
        number = float(value)
        if not number.is_integer():
            raise DataValidationError(f"학번 {value!r}은 정수 형태가 아닙니다.")
        return str(int(number))
    text = str(value).strip()
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return text


def _to_number(value: Any, *, label: str, excel_row: int) -> int | float | None:
    if _is_blank(value):
        return None
    if isinstance(value, bool):
        raise DataValidationError(f"{excel_row}행 {label}에 논리값이 들어 있습니다.")
    if isinstance(value, Integral):
        return int(value)
    if isinstance(value, Real):
        number = float(value)
    else:
        text = str(value).strip().replace(",", "")
        try:
            number = float(text)
        except ValueError as exc:
            raise DataValidationError(
                f"{excel_row}행 {label} 값 {value!r}을 숫자로 변환할 수 없습니다."
            ) from exc
    return int(number) if number.is_integer() else number


## 시험 차수·파일 검색·메타데이터

파일A의 A1에서 연도와 1~6차 시험 번호를 읽고 파일A·파일B를 찾습니다.


In [3]:
def parse_test_number(a1_text: str) -> int:
    match = TEST_NUMBER_PATTERN.search(a1_text)
    if not match:
        raise DataValidationError(f"A1에서 시험 차수를 찾지 못했습니다: {a1_text!r}")
    test_number = int(match.group(1))
    if test_number not in range(1, 7):
        raise DataValidationError(
            f"시험 차수는 1~6차여야 합니다. 추출된 값: {test_number}"
        )
    return test_number


def parse_test_year(a1_text: str) -> int:
    match = YEAR_PATTERN.search(a1_text)
    if not match:
        raise DataValidationError(f"A1에서 시험 연도를 찾지 못했습니다: {a1_text!r}")
    return int(match.group(1))


def _xlsx_files(directory: Path) -> list[Path]:
    if not directory.exists():
        raise FileNotFoundError(f"폴더가 없습니다: {directory}")
    if not directory.is_dir():
        raise NotADirectoryError(f"폴더 경로가 아닙니다: {directory}")
    return sorted(
        path
        for path in directory.glob("*.xlsx")
        if not path.name.startswith("~$")
    )


def find_file_a(source_dir: Path) -> Path:
    candidates = _xlsx_files(source_dir)
    if len(candidates) != 1:
        names = [path.name for path in candidates]
        raise DataValidationError(
            "Source 폴더에는 임시 파일(~$)을 제외한 .xlsx 파일이 정확히 1개여야 "
            f"합니다. 현재 {len(candidates)}개: {names}"
        )
    return candidates[0]


def find_file_b(legacy_dir: Path, *, test_year: int, previous_test_number: int) -> Path:
    keyword = f"{test_year}년 제{previous_test_number}차"
    candidates = [path for path in _xlsx_files(legacy_dir) if keyword in path.name]
    if len(candidates) != 1:
        names = [path.name for path in candidates]
        raise DataValidationError(
            f"Legacy 폴더에서 {keyword!r}가 들어간 .xlsx 파일을 정확히 1개 "
            f"찾아야 합니다. 현재 {len(candidates)}개: {names}"
        )
    return candidates[0]


def _get_only_visible_sheet(workbook):
    visible_sheets = [sheet for sheet in workbook.worksheets if sheet.sheet_state == "visible"]
    if len(visible_sheets) != 1:
        raise DataValidationError(
            f"표시된 worksheet가 정확히 1개여야 합니다. 현재 {len(visible_sheets)}개입니다."
        )
    return visible_sheets[0]


def read_metadata(file_path: Path) -> SourceMetadata:
    workbook = load_workbook(file_path, data_only=True, read_only=False)
    sheet = _get_only_visible_sheet(workbook)
    a1_text = _normalize_text(sheet["A1"].value)
    if not a1_text:
        raise DataValidationError("파일A의 A1이 비어 있습니다.")
    a2_text = _normalize_text(sheet["A2"].value) or ""
    metadata = SourceMetadata(
        file_path=file_path,
        a1_text=a1_text,
        a2_text=a2_text,
        sheet_title=sheet.title,
        test_year=parse_test_year(a1_text),
        test_number=parse_test_number(a1_text),
    )
    workbook.close()
    return metadata


def _area_column_map(sheet, *, start_column: int) -> dict[str, int]:
    mapping: dict[str, int] = {}
    for column in range(start_column, start_column + 6):
        header = _normalize_text(sheet.cell(4, column).value)
        if header in mapping:
            raise DataValidationError(f"영역명이 중복되어 있습니다: {header!r}")
        if header:
            mapping[header] = column
    missing = [name for name in AREA_COLUMNS if name not in mapping]
    unexpected = [name for name in mapping if name not in AREA_COLUMNS]
    if missing or unexpected:
        raise DataValidationError(
            f"영역 열 구성이 올바르지 않습니다. 누락={missing}, 예상 밖={unexpected}"
        )
    return mapping


def _validate_required_identity(record: dict[str, Any], *, excel_row: int) -> None:
    missing = [name for name in ("학급명", "학생명", "학번") if _is_blank(record[name])]
    if missing:
        raise DataValidationError(f"{excel_row}행 필수값이 비어 있습니다: {missing}")


def _validate_unique_ids(data: pd.DataFrame, *, label: str) -> None:
    duplicate_mask = data["학번"].duplicated(keep=False)
    if duplicate_mask.any():
        duplicate_ids = sorted(data.loc[duplicate_mask, "학번"].unique().tolist())
        raise DataValidationError(f"{label}에 중복 학번이 있습니다: {duplicate_ids}")


def _validate_score_sum(record: dict[str, Any], *, score_column: str, excel_row: int) -> None:
    total = record.get(score_column)
    area_values = [record.get(name) for name in AREA_COLUMNS]
    if total is None or any(value is None for value in area_values):
        return
    if abs(float(total) - sum(float(value) for value in area_values)) > 1e-9:
        raise DataValidationError(
            f"{excel_row}행의 {score_column}({total})과 영역 점수 합계"
            f"({sum(area_values)})가 다릅니다."
        )


## 파일A 정제

일반평가와 누적평가의 서로 다른 열 구조를 읽어 표준 열 순서로 정제합니다.


In [4]:
def _read_normal_file(sheet, *, test_number: int) -> pd.DataFrame:
    area_columns = _area_column_map(sheet, start_column=9)  # I:N
    records: list[dict[str, Any]] = []
    for excel_row in range(5, sheet.max_row + 1):
        selected_values = [sheet.cell(excel_row, column).value for column in (1, 2, 6, 8)]
        selected_values.extend(
            sheet.cell(excel_row, area_columns[name]).value for name in AREA_COLUMNS
        )
        if all(_is_blank(value) for value in selected_values):
            continue
        record = {
            "학급명": _normalize_text(sheet.cell(excel_row, 1).value),
            "학생명": _normalize_text(sheet.cell(excel_row, 2).value),
            "학번": _normalize_student_id(sheet.cell(excel_row, 6).value),
            "점수": _to_number(sheet.cell(excel_row, 8).value, label="점수", excel_row=excel_row),
        }
        for name in AREA_COLUMNS:
            record[name] = _to_number(
                sheet.cell(excel_row, area_columns[name]).value,
                label=name,
                excel_row=excel_row,
            )
        _validate_required_identity(record, excel_row=excel_row)
        _validate_score_sum(record, score_column="점수", excel_row=excel_row)
        records.append(record)
    data = pd.DataFrame(records, columns=NORMAL_BASE_COLUMNS)
    if data.empty:
        raise DataValidationError(f"제{test_number}차 파일에서 학생 데이터를 찾지 못했습니다.")
    _validate_unique_ids(data, label="파일A")
    return data


def _cumulative_exam_columns(test_number: int) -> list[str]:
    if test_number == 3:
        return ["1차", "2차", "3차"]
    if test_number == 6:
        return ["4차", "5차", "6차"]
    raise DataValidationError(f"누적평가 형식은 제3차와 제6차에만 적용됩니다: {test_number}")


def _read_cumulative_file(sheet, *, test_number: int) -> pd.DataFrame:
    exam_columns = _cumulative_exam_columns(test_number)
    metric_columns = [*exam_columns, "주간테스트", "과제수행", "합산점수"]
    metric_map: dict[str, int] = {}
    for column in range(8, 14):  # H:M
        header = _normalize_text(sheet.cell(3, column).value)
        if header:
            metric_map[header] = column
    missing_metrics = [name for name in metric_columns if name not in metric_map]
    if missing_metrics:
        raise DataValidationError(f"누적평가 열이 누락되어 있습니다: {missing_metrics}")

    area_columns = _area_column_map(sheet, start_column=14)  # N:S
    output_columns = ["학급명", "학생명", "학번", *metric_columns, *AREA_COLUMNS]
    records: list[dict[str, Any]] = []
    for excel_row in range(5, sheet.max_row + 1):
        selected_values = [sheet.cell(excel_row, column).value for column in (1, 2, 6)]
        selected_values.extend(sheet.cell(excel_row, metric_map[name]).value for name in metric_columns)
        selected_values.extend(
            sheet.cell(excel_row, area_columns[name]).value for name in AREA_COLUMNS
        )
        if all(_is_blank(value) for value in selected_values):
            continue
        record = {
            "학급명": _normalize_text(sheet.cell(excel_row, 1).value),
            "학생명": _normalize_text(sheet.cell(excel_row, 2).value),
            "학번": _normalize_student_id(sheet.cell(excel_row, 6).value),
        }
        for name in metric_columns:
            record[name] = _to_number(
                sheet.cell(excel_row, metric_map[name]).value,
                label=name,
                excel_row=excel_row,
            )
        for name in AREA_COLUMNS:
            record[name] = _to_number(
                sheet.cell(excel_row, area_columns[name]).value,
                label=name,
                excel_row=excel_row,
            )
        _validate_required_identity(record, excel_row=excel_row)
        _validate_score_sum(record, score_column=f"{test_number}차", excel_row=excel_row)
        records.append(record)
    data = pd.DataFrame(records, columns=output_columns)
    if data.empty:
        raise DataValidationError(f"제{test_number}차 파일에서 학생 데이터를 찾지 못했습니다.")
    _validate_unique_ids(data, label="파일A")
    return data


def read_file_a(file_path: Path, metadata: SourceMetadata) -> pd.DataFrame:
    workbook = load_workbook(file_path, data_only=True, read_only=False)
    sheet = _get_only_visible_sheet(workbook)
    if metadata.test_number in (3, 6):
        data = _read_cumulative_file(sheet, test_number=metadata.test_number)
        expected_degree = 15
    else:
        data = _read_normal_file(sheet, test_number=metadata.test_number)
        expected_degree = 10
    workbook.close()
    if len(data.columns) != expected_degree:
        raise DataValidationError(
            f"정제 직후 degree가 {expected_degree}여야 하지만 {len(data.columns)}입니다."
        )
    return data


## 파일B 이전점수 결합

제2·4·5차에 학번 기준으로 이전점수를 결합합니다. 제4차는 제3차 파일의 `3차` 열을 사용합니다.


In [5]:
def read_previous_scores(file_b: Path, *, previous_test_number: int) -> pd.DataFrame:
    workbook = load_workbook(file_b, data_only=True, read_only=False)
    sheet = _get_only_visible_sheet(workbook)
    if previous_test_number == 3:
        target_header = "3차"
        score_candidates = range(8, 14)  # H:M
        header_row = 3
    else:
        target_header = "점수"
        score_candidates = range(1, sheet.max_column + 1)
        header_row = 3

    score_column = None
    for column in score_candidates:
        if _normalize_text(sheet.cell(header_row, column).value) == target_header:
            score_column = column
            break
    if score_column is None:
        workbook.close()
        raise DataValidationError(
            f"파일B에서 이전점수 원본 열 {target_header!r}을 찾지 못했습니다."
        )

    records: list[dict[str, Any]] = []
    for excel_row in range(5, sheet.max_row + 1):
        row_has_data = any(
            not _is_blank(sheet.cell(excel_row, column).value)
            for column in range(1, min(sheet.max_column, 19) + 1)
        )
        if not row_has_data:
            continue
        student_id = _normalize_student_id(sheet.cell(excel_row, 6).value)
        if student_id is None:
            workbook.close()
            raise DataValidationError(f"파일B의 {excel_row}행에 데이터는 있지만 학번이 없습니다.")
        records.append(
            {
                "학번": student_id,
                "이전점수": _to_number(
                    sheet.cell(excel_row, score_column).value,
                    label=target_header,
                    excel_row=excel_row,
                ),
            }
        )
    workbook.close()
    previous_data = pd.DataFrame(records, columns=["학번", "이전점수"])
    _validate_unique_ids(previous_data, label="파일B")
    return previous_data


def add_previous_score(
    current_data: pd.DataFrame,
    *,
    legacy_dir: Path,
    test_year: int,
    test_number: int,
) -> pd.DataFrame:
    if test_number not in (2, 4, 5):
        return current_data
    previous_test_number = test_number - 1
    file_b = find_file_b(
        legacy_dir,
        test_year=test_year,
        previous_test_number=previous_test_number,
    )
    previous_data = read_previous_scores(
        file_b,
        previous_test_number=previous_test_number,
    )
    merged = current_data.merge(
        previous_data,
        on="학번",
        how="left",
        sort=False,
        validate="one_to_one",
    )
    expected_columns = [*NORMAL_BASE_COLUMNS, "이전점수"]
    merged = merged[expected_columns]
    if len(merged.columns) != 11:
        raise DataValidationError(f"이전점수 추가 후 degree가 11이 아닙니다: {len(merged.columns)}")
    return merged


def clean_file_a(file_a: Path, *, legacy_dir: Path) -> tuple[SourceMetadata, pd.DataFrame]:
    metadata = read_metadata(file_a)
    cleaned_data = read_file_a(file_a, metadata)
    cleaned_data = add_previous_score(
        cleaned_data,
        legacy_dir=legacy_dir,
        test_year=metadata.test_year,
        test_number=metadata.test_number,
    )
    return metadata, cleaned_data


## 반별 Excel 저장

학급명으로 나누고 학생명 오름차순으로 정렬하여 row 3 헤더 형식의 반별 Excel을 저장합니다.


In [ ]:
def _safe_filename_component(value: str) -> str:
    sanitized = INVALID_FILENAME_CHARS.sub("_", value).strip().rstrip(".")
    if not sanitized:
        raise DataValidationError(f"파일명으로 사용할 수 없는 학급명입니다: {value!r}")
    return sanitized


def _excel_value(value: Any) -> Any:
    if _is_blank(value):
        return None
    if hasattr(value, "item"):
        value = value.item()
    return value


def _style_output_sheet(sheet, headers: list[str], data_row_count: int) -> None:
    last_column = len(headers)
    last_column_letter = get_column_letter(last_column)
    sheet.auto_filter.ref = f"A3:{last_column_letter}{3 + data_row_count}"
    sheet.freeze_panes = "A4"

    title_fill = PatternFill("solid", fgColor="DCE6F1")
    header_fill = PatternFill("solid", fgColor="D9EAF7")
    thin_gray = Side(style="thin", color="B7C9D6")
    header_border = Border(bottom=thin_gray)
    sheet["A1"].font = Font(name="맑은 고딕", size=12, bold=True)
    sheet["A1"].fill = title_fill
    sheet["A2"].font = Font(name="맑은 고딕", size=9, color="666666")

    for cell in sheet[3]:
        if cell.column <= last_column:
            cell.font = Font(name="맑은 고딕", size=10, bold=True)
            cell.fill = header_fill
            cell.border = header_border
            cell.alignment = Alignment(horizontal="center", vertical="center")

    widths = {
        "학급명": 27,
        "학생명": 12,
        "학번": 12,
        "주간테스트": 13,
        "과제수행": 12,
        "합산점수": 12,
    }
    for index, header in enumerate(headers, start=1):
        sheet.column_dimensions[get_column_letter(index)].width = widths.get(header, 14)

    id_column = headers.index("학번") + 1
    for row in range(4, 4 + data_row_count):
        sheet.cell(row, id_column).number_format = "@"
        sheet.cell(row, id_column).alignment = Alignment(horizontal="center")
    for index, header in enumerate(headers, start=1):
        if header in {"학급명", "학생명", "학번"}:
            continue
        number_format = "0.0" if header in {"주간테스트", "과제수행", "합산점수"} else "0"
        for row in range(4, 4 + data_row_count):
            sheet.cell(row, index).number_format = number_format
            sheet.cell(row, index).alignment = Alignment(horizontal="right")


def save_class_workbooks(
    cleaned_data: pd.DataFrame,
    *,
    metadata: SourceMetadata,
    output_dir: Path,
    teacher_suffix: str = TEACHER_SUFFIX,
    overwrite_existing: bool = True,
) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    output_files: list[Path] = []
    class_names = sorted(
        cleaned_data["학급명"].dropna().unique().tolist(),
        key=lambda value: unicodedata.normalize("NFC", str(value)),
    )
    for class_name in class_names:
        class_data = cleaned_data.loc[cleaned_data["학급명"] == class_name].copy()
        class_data = class_data.sort_values(
            "학생명",
            key=lambda series: series.map(lambda value: unicodedata.normalize("NFC", str(value))),
            kind="stable",
        )
        base_name = str(class_name).removesuffix(teacher_suffix)
        output_name = f"[{metadata.test_year}-{metadata.test_number}차]{_safe_filename_component(base_name)}-정제.xlsx"
        output_path = output_dir / output_name
        if output_path.exists() and not overwrite_existing:
            raise FileExistsError(f"출력 파일이 이미 있습니다: {output_path}")

        workbook = Workbook()
        sheet = workbook.active
        sheet.title = metadata.sheet_title[:31]
        sheet["A1"] = metadata.a1_text
        sheet["A2"] = metadata.a2_text
        headers = class_data.columns.tolist()
        for column, header in enumerate(headers, start=1):
            sheet.cell(3, column, header)
        for row_offset, row in enumerate(class_data.itertuples(index=False, name=None), start=4):
            for column, value in enumerate(row, start=1):
                sheet.cell(row_offset, column, _excel_value(value))
        _style_output_sheet(sheet, headers, len(class_data))
        workbook.save(output_path)
        workbook.close()
        output_files.append(output_path)
    return output_files


## 전체 파이프라인

파일 검색부터 반별 저장까지 한 번에 실행하는 함수입니다.


In [7]:
def run_pipeline(
    *,
    source_dir: str | Path,
    legacy_dir: str | Path,
    output_dir: str | Path,
    teacher_suffix: str = TEACHER_SUFFIX,
    overwrite_existing: bool = True,
) -> CleaningResult:
    source_dir = Path(source_dir)
    legacy_dir = Path(legacy_dir)
    output_dir = Path(output_dir)
    file_a = find_file_a(source_dir)
    metadata, cleaned_data = clean_file_a(file_a, legacy_dir=legacy_dir)
    output_files = save_class_workbooks(
        cleaned_data,
        metadata=metadata,
        output_dir=output_dir,
        teacher_suffix=teacher_suffix,
        overwrite_existing=overwrite_existing,
    )

    print(f"파일A: {file_a.name}")
    print(f"시험 차수: 제{metadata.test_number}차")
    print(f"정제 학생 수: {len(cleaned_data)}명")
    print(f"저장한 반별 파일 수: {len(output_files)}개")
    for output_path in output_files:
        print(f"- {output_path.name}")

    return CleaningResult(
        metadata=metadata,
        cleaned_data=cleaned_data,
        output_files=output_files,
    )


## 경로 설정

필요한 경우 아래 세 경로와 교사명 suffix만 수정하세요. `RUN_PIPELINE`을 `True`로 두고 전체 셀을 실행하면 저장까지 진행됩니다.


In [8]:

SOURCE_DIR = Path(r"C:\Users\9191h\Desktop\TestFeedback\Source")
LEGACY_DIR = Path(r"C:\Users\9191h\Desktop\TestFeedback\Source\Legacy")
OUTPUT_DIR = Path(r"C:\Users\9191h\Desktop\TestFeedback\DataClean")

TEACHER_SUFFIX_FOR_FILENAME = "-임서영T"
OVERWRITE_EXISTING = True
RUN_PIPELINE = True


## 실행

오류가 발생하면 파일을 저장하지 않고 원인을 한국어 메시지로 표시합니다. 정상 완료되면 생성된 반별 파일명과 학생 수가 출력됩니다.


In [9]:

result = None

if RUN_PIPELINE:
    result = run_pipeline(
        source_dir=SOURCE_DIR,
        legacy_dir=LEGACY_DIR,
        output_dir=OUTPUT_DIR,
        teacher_suffix=TEACHER_SUFFIX_FOR_FILENAME,
        overwrite_existing=OVERWRITE_EXISTING,
    )
    display(result.cleaned_data.head())
else:
    print("RUN_PIPELINE=False이므로 파일을 처리하지 않았습니다.")


파일A: 성적현황 시험지_학생별(기파랑문해원 2026년 제2차 정기평가)_20260807000526.xlsx
시험 차수: 제2차
정제 학생 수: 36명
저장한 반별 파일 수: 5개
- 선랑-금1830-정제.xlsx
- 선랑-일0930-정제.xlsx
- 선랑-토0930-정제.xlsx
- 선랑-토1610-정제.xlsx
- 향도-일1310-정제.xlsx


,학급명,학생명,학번,점수,어휘 파악,핵심 요지 파악,내용 파악,문맥 파악,구조 파악,추론,이전점수
0,선랑-토0930-임서영T,김하민,46861,73,17,7,15,9,6,19,47.0
1,선랑-일0930-임서영T,서현우,61691,73,17,3,12,12,9,20,44.0
2,선랑-토1610-임서영T,김세현,26292,72,10,7,15,12,12,16,47.0
3,선랑-금1830-임서영T,박세은,59051,70,15,3,12,9,12,19,NaN
4,향도-일1310-임서영T,노윤준,58852,69,17,3,18,6,9,16,72.0


## 적용된 이전점수 규칙

| 파일A 시험 차수 | 이전점수 처리 |
|---|---|
| 1차 | 이전점수 없음 |
| 2차 | 제1차 파일B의 `점수` |
| 3차 | 누적평가 형식, 이전점수 열 없음 |
| 4차 | 제3차 파일B의 `3차` |
| 5차 | 제4차 파일B의 `점수` |
| 6차 | 누적평가 형식, 이전점수 열 없음 |
